# W3 실습 — 같은 모델, 네 가지 무게

**Potato LLM 3주차 (8/5)**

## 오늘 하는 것

| | 내용 |
|---|---|
| ① | **정보 이론에서 출발**해 — 비트, 엔트로피, 손실 압축 |
| ② | **MDL**로 "모델 가중치에는 압축 여지가 있다"는 직관을 얻고 |
| ③ | 그 직관을 **양자화**로 실행해, 같은 모델을 4가지 무게(fp16 · Q8 · Q4 · Q2)로 눌러본다 |
| ④ | W2 시험지로 **속도 · 메모리 · 점수 3축**을 재서 무너지는 지점을 찾는다 |

**④가 오늘의 핵심입니다.** "양자화하면 빨라진다"를 말로 듣는 게 아니라,
**어디까지 줄여도 점수가 버티는지** 내 손으로 확인하는 거예요.

## 쓰는 법

- W2와 같습니다. 셀을 위에서부터 차례로 실행하세요. `Shift + Enter`
- 바꿀 곳에는 **`← 여기`** 표시가 되어 있어요
- 오류가 나면 **디스코드에 화면을 그대로 캡처해서** 올려주세요

## 준비물

**구글 계정 하나면 됩니다.** 이번 주도 전원 **Colab**으로 진행해요.

> 📢 **다음 주(W4, 8/12)는 오프라인입니다 — 장소: 오픈업 센터.** 중간 공유 + 팀 구성이 있어요. 자세한 공지는 디스코드에서!

## 0. 준비 — W2와 동일

### 먼저 GPU를 켜주세요
상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 `T4 GPU` → 저장**

In [ ]:
!nvidia-smi -L
!apt-get -qq install -y zstd lshw
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5
!curl -s localhost:11434

## 1. 정보를 압축할 수 있을까?

됩니다 — 그리고 우리는 이미 매일 하고 있어요. ZIP과 JPG.

| | 무손실 (ZIP·PNG) | 손실 (JPG·MP3) |
|---|---|---|
| 원리 | 중복만 제거, 복원하면 원본과 동일 | **사람이 덜 민감한 정보부터 버림** — 눈에 안 띄는 색, 귀에 안 들리는 주파수 |
| 한계 | 줄일 수 있는 한계가 정해져 있음 | 훨씬 작게 가능 — 잘 버리면 티도 안 남 |

압축이 되는 이유는 하나입니다: **데이터의 상당 부분은 중복이거나, 없어도 티가 안 나는 것** — 즉, 대부분이 비어 있기 때문이에요.

그렇다면 오늘의 첫 질문 — **모델(가중치)도 이렇게 비어 있을까요?**

## 2. 모델도 비어 있다 — 왜?

학습된 모델의 상태를 세 가지로 나눠 봅시다:

| | 모델 상태 | 결과 |
|---|---|---|
| **외운다** | 정답을 통째로 기억 → 모델이 **복잡**해지고 설명 가능성이 낮아짐 | **과적합(오버피팅)** — 훈련 데이터엔 강한데 새 데이터에 약함 |
| **일반화한다** | 패턴만 남기고 나머지는 버림 → 모델이 **간단**해짐 | **우리가 원하는 균형점** |
| **너무 단순하다** | 패턴까지 버림 | **언더피팅** — 훈련 데이터조차 설명 못 함 |

실제 학습이 끝난 가중치에는 이 스펙트럼의 흔적이 섞여 있습니다 — **외운 흔적, 노이즈, 쓰지 않는 정밀도.** 게다가 모든 정밀도가 추론에 똑같이 중요하지도 않아요. **모델이 비어 있는 이유입니다.**

그럼 다음 질문 — **"비어 있다"는 걸, 숫자로 잴 수 있을까요?** 잴 수 있습니다. 그 자가 계보의 첫 정거장, 정보 이론입니다.

## 3. 정보 이론 — 압축 가능량을 재는 자

### 3-1. 비트 = 표현할 수 있는 경우의 수

모델의 가중치 하나는 결국 **비트 몇 개로 저장된 숫자**입니다. 비트 n개가 표현할 수 있는 값은 정확히 **2ⁿ가지**.

| 비트 | 표현 가능한 값 | |
|---|---|---|
| 16bit | 65,536 가지 | |
| 8bit | 256 가지 | 256분의 1로 해상도 감소 |
| 4bit | **16 가지** | 4,096분의 1 |
| 2bit | **4 가지** | 가중치 하나가 딱 4단계 |

65,536가지가 담던 정보를 16가지에 다 담을 수는 없습니다(비둘기집 원리). **비트를 줄이면 정보는 반드시 사라져요.**

> 파일 크기 환산: 16bit = 2byte → 3B개 × 2byte ≈ 6.2GB, 4bit = 0.5byte → ≈ 1.5GB.
> 실제 Q4 파일이 1.9GB인 이유: scale 등 **메타데이터가 추가**되기 때문 — 크기가 정확히 1/4로 떨어지지는 않습니다.

### 3-2. 정보량 — 놀라움을 비트로 재다

사건 하나가 일어났을 때 그 소식이 담는 정보의 양:

$$I(x) = -\log_2 p(x) \ \text{비트}$$

- 직관: **드문 일일수록 놀랍고, 놀라울수록 정보가 많다**
- 확률 1짜리 사건(항상 앞면): $I = 0$비트 — 뻔한 소식은 전달할 정보가 없음
- 확률 ½짜리 사건(동전 앞면): $I = 1$비트
- 확률 1/64짜리 사건: $I = 6$비트 — "그런 일이?!"를 전달하려면 6비트가 필요

왜 log냐면: **스무고개**로 생각하면 됩니다. 64가지 중 하나를 맞히려면 예/아니오 질문이 $\log_2 64 = 6$번 필요하죠. 정보량 = "이 사건을 특정하는 데 필요한 질문(비트) 수"입니다.

### 3-3. 엔트로피 — 놀라움의 평균, 압축의 바닥

*Claude Shannon, "A Mathematical Theory of Communication" (1948) — 정보 이론의 창시 논문*

사건 하나가 아니라 **확률변수 전체의 평균 정보량**입니다:

$$H(X) = -\sum_x p(x)\,\log_2 p(x)$$

$I$가 "이번 소식 하나의 무게"라면, $H$는 각 사건의 $I(x)$를 확률로 가중평균한 **"이 소식통의 평균 무게"**입니다.

- 공평한 동전: 앞·뒤 각각 $I=1$비트 → 평균도 $H=1$비트 · 공평한 주사위: $H=\log_2 6 \approx 2.585$비트
- 찌그러진 동전(앞면 99%): 뒷면은 $I \approx 6.6$비트로 무겁지만 드물게 옴 → 가중평균 $H \approx 0.08$비트 — **뻔한 소식통은 평균이 가볍다**

섀넌의 정리(무손실 부호화의 하한): "자주 나오는 건 짧게" 요령을 다 부려도 **어떤 데이터도 평균적으로 자기 엔트로피 $H$ 밑으로는 무손실 압축이 불가능합니다.** 바닥을 알아야, 그 밑으로 내려가는 일이 '요령'이 아니라 '희생'임을 알 수 있어요. (덤: LLM 평가에 쓰는 perplexity가 바로 $2^{H}$ — 이 엔트로피 계열 지표입니다.)

그럼 이 "자"를 **모델의 압축**에 들이댄 사람이 있을까요? — 있습니다.

## 4. MDL — 압축 관점으로 학습을 다시 쓰다

2장의 균형(외움 ↔ 일반화 ↔ 단순)을 **비트 수로 정식화**한 것이 **MDL(최소 서술 길이) 원리**입니다 — Rissanen(1978)이 처음 제안했고, 신경망에 적용한 고전이 **Hinton & van Camp (1993), "Keeping Neural Networks Simple by Minimizing the Description Length of the Weights"** 입니다:

$$L(\text{전체}) = \underbrace{L(\text{데이터} \mid \text{모델})}_{\text{오차를 설명하는 비용}} + \underbrace{L(\text{모델})}_{\text{가중치를 설명하는 비용}}$$

- 외우면: 오차 비용은 0에 가깝지만 가중치 비용이 폭발 → 합이 커짐
- 너무 단순하면: 가중치 비용은 작지만 오차 비용이 폭발 → 합이 커짐
- **일반화 = 이 합이 최소가 되는 균형점.** "가중치에 담긴 정보량이 적을수록 잘 일반화된다"

논문은 이를 실제 학습 목적함수로 만들었습니다 — 각 가중치를 분포 $Q$로 두고:

$$\text{Cost} = \mathbb{E}_{w \sim Q}\big[\text{데이터 오차}\big] + \mathrm{KL}\big(Q \,\|\, P\big)$$

$\mathrm{KL}(Q\|P)$가 곧 "가중치에 담긴 정보량(비트)"입니다. (이 구조가 훗날 변분 추론·VAE의 ELBO로 이어집니다.)

### 4-1. MDL과 양자화 — 같은 생각, 다른 방법

| | 같은 것 | 다른 것 |
|---|---|---|
| **철학** | **"가중치의 정보량을 줄이는 것은 좋다"** — 일반화의 방향과 압축의 방향이 같다. MDL이 양자화·프루닝의 개념적 근거 | |
| **방법·시점** | | **MDL(1993)**: 학습하면서 비트를 줄임 (분포 + KL 최적화) · **양자화(오늘)**: 학습 끝난 모델을 반올림으로 누름 (사후 압축) |

**정직한 주의**: 방법이 다르기 때문에, MDL은 "학습 끝난 모델을 Q4로 눌러도 점수가 보존된다"를 **보장하지 않습니다.** 이론이 주는 건 "압축 여지가 있다"는 직관까지 — **어디까지 눌러도 되는지는 재봐야 압니다.** 그게 오늘 실습입니다.

## 5. 그래서 양자화 — 모델의 손실 압축

모델의 몸무게는 **파라미터 수 × 파라미터 하나의 크기**입니다.
파라미터 수(3B)는 못 줄여도, **하나하나의 크기(비트)는 줄일 수 있어요.**
16비트 소수를 8·4·2비트 정수로 **반올림해서 저장**하는 것 — 모델에 적용한 JPG입니다.

| 표기 | 파라미터 하나 | 3B 모델 크기 | 비유 |
|---|---|---|---|
| fp16 | 16bit | ~6.2GB | 원본 사진 |
| Q8 | 8bit | ~3.3GB | 고화질 JPG |
| **Q4** | 4bit | ~1.9GB | 보통 JPG — **Ollama 기본값이 이겁니다** |
| Q2 | 2bit | ~1.3GB | 심하게 압축한 JPG |

> W2에서 여러분이 받은 `qwen2.5:3b`는 사실 **이미 Q4_K_M로 양자화된 모델**이었어요.
> 오늘은 그 원본(fp16)과 더 눌러버린 것(Q2)까지 나란히 세워봅니다.

덤: 추론 속도는 대체로 빨라집니다 — LLM 추론이 메모리 읽기에 묶여 있어서(memory-bound), 파일이 작아진 만큼 빨리 읽거든요. 다만 하드웨어·커널에 따라 달라서 **보장은 아닙니다.**

### 5-1. 원리 — 대표 16개로 바꿔치기

복잡해 보이지만, 양자화가 하는 일은 한 문장입니다:

> **서로 다른 숫자 30억 개를, 미리 정한 대표값 16개 중 가장 가까운 것으로 바꾼다.**

숫자들이 흩어져 있는 범위(min~max)에 눈금 16개를 긋고, 각 숫자를 제일 가까운 눈금에 붙이는 거예요.

**결과적으로 하는 일**: 이제 각 가중치는 정밀한 숫자 대신 **"몇 번 눈금인지"만 기억**하면 됩니다. 번호는 0~15 — **그래서 4비트면 충분한 겁니다.** 16비트 숫자가 4비트 번호가 되니 크기가 ¼. 예: `0.083 → 13번 눈금 → (복원하면) 0.089` — 오차 0.006은 영원히 못 되찾습니다.

**같은 말을 수식으로 쓰면 세 줄입니다** ($b$비트, 칸 수 $2^b$):

1. 눈금 간격은 얼마인가: $\text{scale} = \frac{\max(w) - \min(w)}{2^b - 1}$
2. 몇 번 눈금인가 (반올림 — 정보가 사라지는 유일한 순간): $q_i = \mathrm{round}\!\left(\frac{w_i - \min(w)}{\text{scale}}\right)$
3. 쓸 때는 복원: $\hat{w}_i = q_i \cdot \text{scale} + \min(w)$

덤으로 얻는 공식 하나 — 반올림이니 오차는 칸 폭의 절반을 못 넘습니다: $|w_i - \hat{w}_i| \le \tfrac{\text{scale}}{2}$. **눈금 폭(scale)이 커지면 오차 상한도 커진다** — 이게 바로 다음의 아웃라이어 사건을 예고해요.

아래 셀에서 직접 해보세요.

In [ ]:
import numpy as np

def quantize_4bit(w):
    """4비트(16칸) 양자화: 칸 번호로 저장했다가 복원한다."""
    lo, hi = w.min(), w.max()
    scale = (hi - lo) / 15                      # 16칸이니 경계는 15개
    idx = np.round((w - lo) / scale)            # 0~15 칸 번호로 반올림 ← 정보가 사라지는 순간
    return idx * scale + lo                     # 복원

# 어떤 레이어의 가중치 8개라고 칩시다
w = np.array([0.12, -0.03, 0.08, -0.11, 0.05, 0.01, -0.07, 0.10])
restored = quantize_4bit(w)

print(f"{'원본':>8} {'복원':>8} {'오차':>8}")
for a, b in zip(w, restored):
    print(f"{a:8.3f} {b:8.3f} {abs(a-b):8.3f}")
print(f"\n평균 오차: {abs(w - restored).mean():.4f}")

# 아웃라이어 3.0 하나가 끼면?
w2 = np.append(w, 3.0)
restored2 = quantize_4bit(w2)
print(f"[아웃라이어 포함] 나머지 8개의 평균 오차: {abs(w2[:-1] - restored2[:-1]).mean():.4f}")
print("→ 구간이 -0.11~3.0으로 늘어나 칸 폭이 20배 커짐 → 평범한 값들이 다 뭉개집니다")
print("→ 그래서 실전에서는 가중치를 '블록'으로 잘게 나눠 블록마다 scale을 따로 잡습니다. 그게 K-quant의 'K'예요")

### 5-2. K와 M — 실전 양자화의 두 가지 지혜

**K = 블록별 scale.** 방금 봤듯 아웃라이어 하나가 전체 눈금을 망칩니다. 그래서 가중치를 블록으로 잘게 나눠 블록마다 scale을 따로 잡아요 — 피해가 그 블록 안에 갇힙니다. 이게 llama.cpp K-quant, 현재 GGUF 표준.

**M = 비트 예산 배분의 등급.** 핵심 통찰은 **"학습된 가중치의 모든 정밀도가 추론에 똑같이 중요하지는 않다"** 입니다(4장의 MDL 통찰 그대로). 민감한 블록엔 비트를 더 주고 둔감한 블록에서 아끼는 것 — JPG가 눈에 안 띄는 색부터 버리는 것과 같은 원리예요. `Q4_K_M`의 M(Medium)이 그 등급 표기입니다.

> 실제 GGUF 구현은 블록 크기·메타데이터·혼합 정밀도가 얽혀 이보다 복잡합니다. 오래가는 지식은 이름 해독보다 **"평균 비트 수 외에도 어디에 비트를 쓰느냐가 품질을 좌우한다"** 는 원리 쪽이에요.

**이름 읽는 법** — 허깅페이스에서 GGUF 고를 때 매번 만나는 꼬리표:

| 조각 | 뜻 |
|---|---|
| `q4` | 파라미터를 **4비트**로 (칸 16개) |
| `K` | **블록별 scale** — 아웃라이어 대책 |
| `M` | Medium — 비트 예산 배분의 중간 등급 (S/M/L) |

외울 건 하나 — **모르면 `Q4_K_M`을 고르세요.** 크기·품질 균형의 업계 기본값입니다.

### 5-3. 오늘 실험의 예측 — 왜 도구 호출이 먼저 깨지나

자연어는 **정답이 여러 개인 출력**입니다. "결과를 알려드리겠습니다"와 "결과는 다음과 같습니다"는 다른 문자열이지만 같은 일을 하죠. 확률 분포가 조금 뭉개져도 *괜찮은 다른 표현*으로 샐 수 있어서 손상이 티가 잘 안 납니다.

도구 호출은 반대입니다. JSON에도 구문적 여유가 아주 없진 않지만, **채점이 all-or-nothing**이에요 — 도구명·인자명·타입·형식 중 **하나만 어긋나도 호출 전체가 실패**로 판정됩니다. 오류를 흡수할 완충이 없는 건 출력 자체라기보다 **성공 조건의 구조**이고, 그래서 같은 크기의 손상이 자연어보다 먼저, 그대로 드러납니다.

> **예측**: 무게를 누르면 **자연어 답변은 멀쩡해 보여도 도구 호출부터 깨질 것이다.**
> 아래 실습에서 확인해보세요. — 이건 W6에서 배울 처방(제약 디코딩 = 형식을 문법으로 강제하는 것)의 복선이기도 합니다.

## 6. 같은 모델, 네 가지 무게 받기 `5~8분 · 약 13GB`

같은 Qwen2.5-3B-Instruct를 양자화 수준만 바꿔서 4개 받습니다.

> 다운로드가 오래 걸리면 fp16을 빼고 3개로 하셔도 됩니다. 결과 비교에는 지장 없어요.

In [ ]:
VARIANTS = [                                # ← 시간이 없으면 fp16 줄을 지우세요
    "qwen2.5:3b-instruct-fp16",   # 원본 (~6.2GB)
    "qwen2.5:3b-instruct-q8_0",   # 8bit (~3.3GB)
    "qwen2.5:3b-instruct-q4_K_M", # 4bit (~1.9GB) — W2에서 쓴 그 모델
    "qwen2.5:3b-instruct-q2_K",   # 2bit (~1.3GB)
]

for v in VARIANTS:
    print(f"\n=== {v}")
    !ollama pull {v}

!ollama list

## 7. 시험지 + 채점기 — W2에서 그대로 가져옴

문항도 채점 방식도 W2와 완전히 같습니다. **바뀌는 건 모델의 무게뿐** — 그래야 차이가 양자화 때문이라고 말할 수 있어요. (실험에서 변수는 한 번에 하나!)

In [ ]:
import json, shutil, subprocess, time, urllib.request

OLLAMA = "http://localhost:11434"

TOOLS = [
    {"type": "function", "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {"type": "object", "properties": {
            "city": {"type": "string", "description": "도시 이름 (영문)"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}},
            "required": ["city"]}}},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "파일 내용을 읽는다",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "파일 경로"}},
            "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "수식을 계산한다",
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "계산할 수식, 예: 2+3*4"}},
            "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "send_email",
        "description": "이메일을 보낸다",
        "parameters": {"type": "object", "properties": {
            "to": {"type": "string"}, "subject": {"type": "string"},
            "body": {"type": "string"}},
            "required": ["to", "subject", "body"]}}},
]

CASES = [
    {"q": "서울의 현재 날씨를 섭씨로 알려줘.",
     "tool": "get_weather",
     "check": lambda a: "seoul" in str(a.get("city", "")).lower()},
    {"q": "report.csv 파일의 내용을 읽어줘.",
     "tool": "read_file",
     "check": lambda a: "report.csv" in str(a.get("path", ""))},
    {"q": "1847 곱하기 365는 얼마야? 도구를 써서 정확히 계산해.",
     "tool": "calculator",
     "check": lambda a: "1847" in str(a.get("expression", "")) and "365" in str(a.get("expression", ""))},
    {"q": "mentor@potato-llm.dev 에게 제목 '중간 공유', 내용 '8월 12일 오프라인에서 뵙겠습니다'로 메일 보내줘.",
     "tool": "send_email",
     "check": lambda a: a.get("to") == "mentor@potato-llm.dev" and len(str(a.get("subject", ""))) > 0 and len(str(a.get("body", ""))) > 0},
    {"q": "지금 KOSPI 지수 알려줘.",
     "tool": None, "check": lambda a: True},
]


def run_case(model, case):
    body = json.dumps({
        "model": model, "stream": False,
        "messages": [{"role": "user", "content": case["q"]}],
        "tools": TOOLS,
    }).encode()
    t0 = time.time()
    req = urllib.request.Request(f"{OLLAMA}/api/chat", body, {"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        resp = json.load(r)
    dt = time.time() - t0
    msg = resp.get("message", {})
    calls = msg.get("tool_calls") or []
    tps = resp.get("eval_count", 0) / (resp.get("eval_duration", 1) / 1e9) if resp.get("eval_duration") else 0

    if case["tool"] is None:
        ok = len(calls) == 0
        detail = f"환각! 없는 기능에 도구 사용: {[c['function']['name'] for c in calls]}" if calls else "도구 안 씀 (정답)"
    elif not calls:
        ok, detail = False, "tool call 없음 (그냥 텍스트로 답변)"
    else:
        fn = calls[0]["function"]
        args = fn.get("arguments", {})
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except json.JSONDecodeError:
                return {"ok": False, "detail": f"인자 JSON 파싱 실패: {args[:80]}",
                        "tps": tps, "sec": dt, "raw": msg}
        ok = fn.get("name") == case["tool"] and case["check"](args)
        detail = f"{fn.get('name')}({json.dumps(args, ensure_ascii=False)[:100]})" + ("" if ok else " ← 오답")
    return {"ok": ok, "detail": detail, "tps": tps, "sec": dt, "raw": msg}


def loaded_size(model):
    """ollama ps 로 지금 메모리에 올라간 크기를 읽는다."""
    out = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
    for line in out.splitlines()[1:]:
        if line.split() and line.split()[0] == model:
            parts = line.split()
            return " ".join(parts[2:4])
    return "?"


def show(i, case, r):
    raw = r.get("raw") or {}
    said = (raw.get("content") or "").strip().replace("\n", " ")
    calls = json.dumps(raw.get("tool_calls") or [], ensure_ascii=False)
    print(f"[{'O' if r['ok'] else 'X'}] Q{i} {case['q'][:34]}")
    print(f"      · 말한 것 : {said[:90] or '(없음 — 도구만 불렀습니다)'}")
    print(f"      · 부른 것 : {calls[:120]}")
    print(f"      · 채  점 : {r['detail']}  ({r['sec']:.1f}s, {r['tps']:.0f} tok/s)")


print(f"시험지 {len(CASES)}문항 + 채점기 준비 완료")

## 8. 시험 실행 — 무게별로 나란히 `5~10분`

**실행하기 전에, 먼저 예측을 적어보세요** (디스코드 채팅에):

> "나는 __ 까지는 점수가 버티고, __ 부터 무너진다고 예측한다"

4개 모델이 차례로 시험을 봅니다. 모델을 하나씩 메모리에 올렸다 내리면서 진행돼요.

**보는 포인트**: 무게가 가벼워질수록 ① 메모리가 어떻게 줄고 ② 속도가 어떻게 변하고 ③ **점수는 몇 비트부터 무너지는가** — 내 예측과 맞았나요?

In [ ]:
records = []
for model in VARIANTS:
    print(f"\n{'='*70}\n모델: {model}\n{'='*70}")
    results = []
    for i, case in enumerate(CASES, 1):
        try:
            r = run_case(model, case)
        except Exception as e:
            r = {"ok": False, "detail": f"에러: {e}", "tps": 0, "sec": 0, "raw": {}}
        results.append(r)
        show(i, case, r)
    mem = loaded_size(model)
    n_ok = sum(r["ok"] for r in results)
    tps_list = [r["tps"] for r in results if r["tps"]]
    avg_tps = sum(tps_list) / max(1, len(tps_list))
    print(f"\n  ▶ 성공률 {n_ok}/{len(CASES)} · 평균 {avg_tps:.0f} tok/s · 메모리 {mem}")
    records.append({"model": model, "score": f"{n_ok}/{len(CASES)}", "tps": f"{avg_tps:.0f}", "mem": mem})
    subprocess.run(["ollama", "stop", model], capture_output=True)  # 다음 모델을 위해 메모리 비우기

## 9. 비교표 — 오늘의 산출물

두 가지 주의하고 읽어주세요:
- 이 결과는 **문항 5개짜리 교육용 관찰**입니다. 5/5와 4/5의 차이를 일반화하기엔 표본이 작아요 — 진짜 판단은 문항을 늘려서(여러분의 벤치 기여 20문항이 그것) 해야 합니다.
- 마지막 열 **"내 램에 올라가는가"**가 실무에선 첫 관문입니다. 아무리 점수가 좋아도 안 올라가면 선택지가 아니에요.

In [ ]:
이름 = "홍길동"          # ← 본인 이름
내_램_GB = 16            # ← 본인 노트북 램 용량

print("| 이름 | 모델 | 양자화 | 성공률 | tok/s | 메모리 | 내 램에 올라가나 |")
print("|---|---|---|---|---|---|---|")
for rec in records:
    quant = rec["model"].split("-")[-1]
    try:
        need_gb = float(rec["mem"].split()[0])
        fits = "O" if need_gb < 내_램_GB * 0.7 else ("겨우" if need_gb < 내_램_GB else "X")
    except (ValueError, IndexError):
        fits = "?"
    print(f"| {이름} | qwen2.5:3b | {quant} | {rec['score']} | {rec['tps']} | {rec['mem']} | {fits} |")
print()
print("결론은 'Q4가 정답'이 아니라 — 내 배포 환경과 내 시험지로 [가장 작은 합격 모델]을 찾는 것입니다.")

## 10. 더 해보기

1. **어디부터 무너졌나요?** Q2에서 실패한 문항의 `부른 것`을 보세요. JSON이 깨졌나요(A유형)? 도구를 아예 안 불렀나요(D유형)? — 실패의 *모양*이 W6에서 배울 처방을 결정합니다.
2. **다른 모델도 눌러보세요** — `llama3.2:1b` 같은 작은 모델은 Q4에서도 버틸까요? 작은 모델일수록 양자화 손상이 큰 경향이 있습니다. 직접 확인해보세요.
3. **내 업무 문항으로도** — W2에서 만든 `MY_CASES`를 이 노트북에 붙여넣으면, 내 업무 기준으로도 "몇 비트까지 괜찮은지"를 잴 수 있습니다. 그게 바로 여러분의 **Potato LLM 스펙 결정** 근거가 돼요.

---

### 다음 주 예고 — 오프라인!

**8/12(수) 오픈업 센터**에서 만나요. 중간 공유(라이트닝) + 팀 구성이 있습니다.
오늘 만든 비교표를 들고 오시면 그게 여러분의 라이트닝 발표 자료입니다.

---

### 참고 문헌 (이론부의 출처)

1. **Claude E. Shannon** (1948). *A Mathematical Theory of Communication*. Bell System Technical Journal. — 1장의 비트·엔트로피·무손실 압축 하한(소스 코딩 정리)
2. **Jorma Rissanen** (1978). *Modeling by Shortest Data Description*. Automatica. — MDL 원리의 원조: "좋은 모델 = 데이터를 가장 짧게 서술하게 하는 모델"
3. **Peter Grünwald** (2004). *A Tutorial Introduction to the Minimum Description Length Principle*. — 2장의 "모델 비용 + 오차 비용" 2부 코드 프레임은 이 튜토리얼의 표준 설명
4. **Geoffrey E. Hinton & Drew van Camp** (1993). *Keeping Neural Networks Simple by Minimizing the Description Length of the Weights*. COLT. — MDL을 신경망 학습 목적함수(오차 + KL)로 구현. 변분 추론·VAE의 뿌리
5. llama.cpp **K-quants** (2023, ggerganov/llama.cpp PR #1684) — 3-2장의 블록별 scale·비트 배분이 구현된 곳. 현재 GGUF 양자화 표준

계보로 읽으면: Shannon(정보란 무엇인가) → Rissanen(짧은 서술이 좋은 모델) → Grünwald(정리된 튜토리얼) → Hinton(신경망 적용) → K-quants(오늘 우리가 쓰는 구현).